# Stage 1: Cassava Disease Recognition - Foundation Training

Ce notebook entraîne Qwen3-VL-8B-Instruct sur la reconnaissance des maladies du manioc.

## Objectifs
- Fine-tuner le modèle sur 29,276 images de manioc (dataset équilibré)
- Reconnaissance des 5 classes: CBB, CBSD, CGM, CMD, Healthy
- Utiliser LoRA pour efficacité mémoire
- Optimisation avec Unsloth

## Configuration Matérielle
- GPU: NVIDIA RTX 5090 32GB
- CUDA 12.9
- Precision: bfloat16

## Durée Estimée
- Entraînement: 8-10 heures (5 époques)

---

## 1. Imports et Configuration

In [1]:
import os
import json
import re
import torch
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any
from datasets import Dataset
from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer
)
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers pour optimisations
from unsloth import FastVisionModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/tmp/ipykernel_3629810/529373224.py:18: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


## 2. Configuration des Chemins et Paramètres

In [2]:
# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
DATA_DIR = BASE_DIR / "data" / "stage1"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage1_cassava_foundation"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle
MODEL_NAME = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

# LoRA Configuration
LORA_CONFIG = {
    "r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}

# Training Configuration - Optimisé pour RTX 5090 avec séquences longues
TRAINING_CONFIG = {
    "per_device_train_batch_size": 1,  # Réduit de 2 à 1 pour séquences très longues
    "gradient_accumulation_steps": 8,  # Augmenté de 4 à 8 pour maintenir effective_batch=8
    "num_train_epochs": 5,
    "learning_rate": 2e-5,
    "warmup_steps": 200,
    "logging_steps": 25,
    "save_steps": 500,
    "save_total_limit": 3,
    "fp16": not torch.cuda.is_bf16_supported(),
    "bf16": torch.cuda.is_bf16_supported(),
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "max_grad_norm": 1.0,
    "seed": 42
}

print(f"\nBase Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"\nConfiguration (optimisé pour séquences longues):")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  - Batch size: {TRAINING_CONFIG['per_device_train_batch_size']}")
print(f"  - Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  - Effective batch size: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  - Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  ⚠️  Batch size réduit à 1 pour gérer les séquences VLM très longues (11k+ tokens)")


Base Directory: /home/hounfodjidagba/learning/cassava/new_implementation
Data Directory: /home/hounfodjidagba/learning/cassava/new_implementation/data/stage1
Checkpoint Directory: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_foundation

Configuration (optimisé pour séquences longues):
  - Epochs: 5
  - Batch size: 1
  - Gradient accumulation: 8
  - Effective batch size: 8
  - Learning rate: 2e-05
  ⚠️  Batch size réduit à 1 pour gérer les séquences VLM très longues (11k+ tokens)


## 3. Chargement du Modèle avec Unsloth

Unsloth optimise automatiquement le modèle pour:
- Réduction de la consommation mémoire (jusqu'à 50%)
- Accélération de l'entraînement (2-5x plus rapide)
- Support natif de 4-bit quantization

In [3]:
print("\n" + "="*60)
print("LOADING MODEL WITH UNSLOTH")
print("="*60)

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    target_modules=LORA_CONFIG["target_modules"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

print(f"\nModel: {MODEL_NAME}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable %: {100 * sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters()):.2f}%")
print("="*60)


LOADING MODEL WITH UNSLOTH


==((====))==  Unsloth 2026.1.1: Fast Qwen3_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.



Model: unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit
Trainable parameters: 87,293,952
Total parameters: 5,540,917,488
Trainable %: 1.58%


## 4. Chargement des Datasets Cassava

Nous utilisons les datasets créés par le notebook `00_prepare_cassava_data.ipynb`:
- **Train**: 29,276 images équilibrées
- **Validation**: 3,612 images
- **Test**: 3,619 images (utilisé plus tard pour évaluation finale)

In [4]:
print("\n" + "="*60)
print("LOADING CASSAVA DATASETS")
print("="*60)

def load_json_dataset(file_path):
    """Charge un dataset JSON"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# Charger les datasets cassava
cassava_train = load_json_dataset(DATA_DIR / "cassava_train_balanced.json")
cassava_val = load_json_dataset(DATA_DIR / "cassava_val.json")
cassava_test = load_json_dataset(DATA_DIR / "cassava_test.json")

print(f"Cassava Train (balanced): {len(cassava_train):,} samples")
print(f"Cassava Validation: {len(cassava_val):,} samples")
print(f"Cassava Test: {len(cassava_test):,} samples")
print(f"\nTotal: {len(cassava_train) + len(cassava_val) + len(cassava_test):,} samples")

# Charger les statistiques
stats_path = DATA_DIR / "dataset_stats.json"
if stats_path.exists():
    with open(stats_path, 'r') as f:
        stats = json.load(f)
    print("\nDistribution des classes (Train équilibré):")
    for class_name, count in stats.get('train_balanced_distribution', {}).items():
        percentage = count / len(cassava_train) * 100
        print(f"  {class_name:40s}: {count:5d} ({percentage:5.2f}%)")


LOADING CASSAVA DATASETS
Cassava Train (balanced): 29,276 samples
Cassava Validation: 3,612 samples
Cassava Test: 3,619 samples

Total: 36,507 samples

Distribution des classes (Train équilibré):


## 5. Formatage des Données pour Qwen3-VL

Conversion du format VQA vers le format attendu par Qwen3-VL:
- Extraction des chemins d'images depuis les tags `<img>path</img>`
- Remplacement par `<image>` (marqueur Qwen-VL)
- Mapping des rôles: `user` et `assistant`

In [5]:
def extract_image_path(text):
    """Extrait le chemin de l'image du tag <img>"""
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def format_sample_for_qwen3vl(sample):
    """Prépare un échantillon pour Qwen3-VL"""
    conversations = sample['conversations']
    
    image_path = None
    formatted_conversations = []
    
    for turn in conversations:
        role = turn['from']
        content = turn['value']
        
        if '<img>' in content:
            img_path = extract_image_path(content)
            if img_path:
                image_path = img_path
            # Remplacer <img>path</img> par <image> (marqueur Qwen-VL)
            content = re.sub(r'<img>.*?</img>', '<image>', content)
        
        formatted_conversations.append({
            "role": role,
            "content": content
        })
    
    return {
        "conversations": formatted_conversations,
        "image_path": image_path,
        "id": sample['id']
    }

print("\nFormatting datasets...")
train_formatted = [format_sample_for_qwen3vl(s) for s in tqdm(cassava_train, desc="Train")]
val_formatted = [format_sample_for_qwen3vl(s) for s in tqdm(cassava_val, desc="Val")]

# Créer datasets HuggingFace
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)

print(f"\nTrain dataset: {len(train_dataset):,} samples")
print(f"Eval dataset: {len(eval_dataset):,} samples")

# Afficher un exemple
print("\n--- Exemple formaté ---")
example = train_dataset[0]
print(f"ID: {example['id']}")
print(f"Image path: {example['image_path'][:80]}...")
print(f"Conversations:")
for i, conv in enumerate(example['conversations']):
    print(f"  [{i}] {conv['role']}: {conv['content'][:100]}...")


Formatting datasets...


Val: 100%|██████████| 3612/3612 [00:00<00:00, 597249.31it/s]


Train dataset: 29,276 samples
Eval dataset: 3,612 samples

--- Exemple formaté ---
ID: kaggle_3874993628.jpg
Image path: /home/hounfodjidagba/learning/cassava/data/raw/train_images/3874993628.jpg...
Conversations:
  [0] user: Picture 1: <image>
What cassava disease is shown in this image?...
  [1] assistant: Disease: Cassava Mosaic Disease (CMD)

Cassava Mosaic Disease (CMD) is one of the most devastating c...


## 6. Data Collator pour Qwen3-VL

Le Data Collator gère:
- Chargement des images depuis les chemins absolus
- Application du chat template Qwen3-VL
- Tokenization avec le processor
- Création des labels pour l'entraînement

In [6]:
@dataclass
class Qwen3VLDataCollator:
    """Data collator pour Qwen3-VL avec support cassava"""
    processor: Any
    base_dir: str = str(BASE_DIR.parent)  # /home/hounfodjidagba/learning/cassava
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        
        for feature in features:
            image_path = feature['image_path']
            image = None
            
            if image_path:
                # Les chemins sont déjà absolus depuis le notebook de préparation
                try:
                    if os.path.exists(image_path):
                        image = Image.open(image_path).convert('RGB')
                    else:
                        print(f"Warning: Image not found: {image_path[:80]}...")
                        image = Image.new('RGB', (224, 224), color='white')
                except Exception as e:
                    print(f"Error loading image {image_path[:80]}...: {e}")
                    image = Image.new('RGB', (224, 224), color='white')
            
            # Construire les messages en format chat
            conversations = feature['conversations']
            messages = []
            for turn in conversations:
                role = turn['role']
                content = turn['content']
                
                # Le premier message utilisateur contient l'image
                if role == 'user' and image is not None and len(messages) == 0:
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": content.replace('<image>', '').strip()}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })
            
            batch_messages.append(messages)
        
        # Appliquer le chat template et tokenizer
        batch_texts = []
        for messages in batch_messages:
            text = self.processor.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            batch_texts.append(text)
        
        # Extraire toutes les images
        all_images = []
        for messages in batch_messages:
            for msg in messages:
                if isinstance(msg.get("content"), list):
                    for item in msg["content"]:
                        if item.get("type") == "image":
                            all_images.append(item["image"])
        
        # Tokenizer avec le processor - TRUNCATION DÉSACTIVÉE pour images VLM
        # Les images génèrent beaucoup de tokens visuels (10k+), ne pas tronquer
        batch = self.processor(
            text=batch_texts,
            images=all_images if all_images else None,
            return_tensors="pt",
            padding=True,
            truncation=False  # Désactivé - les images VLM génèrent beaucoup de tokens
        )
        
        batch["labels"] = batch["input_ids"].clone()
        
        return batch

data_collator = Qwen3VLDataCollator(processor=processor, base_dir=str(BASE_DIR.parent))

print("Data collator created")
print(f"Base directory for images: {BASE_DIR.parent}")
print(f"Truncation: DISABLED (images generate many visual tokens)")

Data collator created
Base directory for images: /home/hounfodjidagba/learning/cassava
Truncation: DISABLED (images generate many visual tokens)


## 7. Configuration du Trainer

Configuration Hugging Face Trainer avec:
- Evaluation régulière tous les 500 steps
- Sauvegarde du meilleur modèle basé sur eval_loss
- Optimisation AdamW 8-bit
- Cosine learning rate scheduler

In [7]:
print("\n" + "="*60)
print("CONFIGURING TRAINER")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_steps=TRAINING_CONFIG["warmup_steps"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=TRAINING_CONFIG["save_total_limit"],
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["save_steps"],
    fp16=TRAINING_CONFIG["fp16"],
    bf16=TRAINING_CONFIG["bf16"],
    optim=TRAINING_CONFIG["optim"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    seed=TRAINING_CONFIG["seed"],
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

effective_batch_size = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch_size * TRAINING_CONFIG['num_train_epochs']

print(f"\nTraining Configuration:")
print(f"  - Device batch size: {TRAINING_CONFIG['per_device_train_batch_size']}")
print(f"  - Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  - Effective batch size: {effective_batch_size}")
print(f"  - Total training steps: {total_steps:,}")
print(f"  - Warmup steps: {TRAINING_CONFIG['warmup_steps']}")
print(f"  - Logging every: {TRAINING_CONFIG['logging_steps']} steps")
print(f"  - Evaluation every: {TRAINING_CONFIG['save_steps']} steps")
print(f"  - Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Precision: {'bfloat16' if TRAINING_CONFIG['bf16'] else 'float16'}")
print("="*60)


CONFIGURING TRAINER

Training Configuration:
  - Device batch size: 1
  - Gradient accumulation: 8
  - Effective batch size: 8
  - Total training steps: 18,295
  - Warmup steps: 200
  - Logging every: 25 steps
  - Evaluation every: 500 steps
  - Learning rate: 2e-05
  - Precision: bfloat16


/tmp/ipykernel_3629810/2291968853.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer._unsloth___init__`. Use `processing_class` instead.
  trainer = Trainer(


## 8. Entraînement

⚠️ **ATTENTION**: Cette cellule peut prendre 8-10 heures sur RTX 5090.

Le modèle sera sauvegardé automatiquement:
- Tous les 500 steps dans `checkpoints/stage1_cassava_foundation/checkpoint-XXXX/`
- Le meilleur modèle sera chargé à la fin

**Objectifs**:
- Précision globale: >85%
- Précision CBB (classe rare): >75%
- Précision CMD (classe dominante): >90%

In [8]:
print("\n" + "="*60)
print("STARTING TRAINING - STAGE 1: CASSAVA DISEASE RECOGNITION")
print("="*60)
print(f"Estimated time: 8-10 hours on RTX 5090")
print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(eval_dataset):,}")
print(f"Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print("="*60)

# Lancer l'entraînement
trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 29,276 | Num Epochs = 5 | Total steps = 18,300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 87,293,952 of 8,854,417,648 (0.99% trained)



STARTING TRAINING - STAGE 1: CASSAVA DISEASE RECOGNITION
Estimated time: 8-10 hours on RTX 5090
Training samples: 29,276
Validation samples: 3,612
Epochs: 5
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
500,7.119100,6.824004
1000,6.987800,6.822987
1500,7.106200,6.822869
2000,7.092500,6.822642


Unsloth: Not an error, but Qwen3VLForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_005/uuid_050e595c-1c60-4ceb-8ca5-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_003/uuid_5a8bfda6-3611-4da9-86f9-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_0adc005b-d4ee-4700-b7ef-...: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_0adc005b-d4ee-4700-b7ef-85c8eff62d54.jpg'
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_005/uuid_050e595c-1c60-4ceb-8ca5-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_003/uuid_5a8bfda6-3611-4da9-86f9-...: broken data stream when reading image file
Error loading image /home/hounfodjidagba/learning/cassava/data/cmd_002/uuid_2e71b426-3fd4-4320-a615-...: cannot identify image file '/home/hounfodjidagba/learning/cassava/data/c

KeyboardInterrupt: 

## 9. Sauvegarde du Modèle Final

In [ ]:
print("\n" + "="*60)
print("SAVING FINAL MODEL")
print("="*60)

# Sauvegarder le modèle complet
final_model_path = CHECKPOINT_DIR / "final_model"
trainer.save_model(str(final_model_path))
tokenizer.save_pretrained(str(final_model_path))

# Sauvegarder les adapters LoRA séparément
lora_adapters_path = CHECKPOINT_DIR / "lora_adapters"
model.save_pretrained(str(lora_adapters_path))

# Sauvegarder les métriques d'entraînement
metrics = {
    "stage": "stage1_cassava_foundation",
    "model": MODEL_NAME,
    "train_samples": len(train_dataset),
    "eval_samples": len(eval_dataset),
    "epochs": TRAINING_CONFIG["num_train_epochs"],
    "lora_config": LORA_CONFIG,
    "training_config": TRAINING_CONFIG,
    "final_eval_loss": trainer.state.log_history[-1].get('eval_loss', None) if trainer.state.log_history else None,
}

with open(CHECKPOINT_DIR / "training_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n✓ Modèle complet sauvegardé: {final_model_path}")
print(f"✓ Adapters LoRA sauvegardés: {lora_adapters_path}")
print(f"✓ Métriques sauvegardées: {CHECKPOINT_DIR / 'training_metrics.json'}")
print("\n" + "="*60)
print("STAGE 1 COMPLETED SUCCESSFULLY")
print("="*60)

## 10. Résumé et Prochaines Étapes

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)

# Afficher l'historique d'entraînement
if trainer.state.log_history:
    print("\nDernières métriques:")
    for entry in trainer.state.log_history[-5:]:
        step = entry.get('step', 'N/A')
        loss = entry.get('loss', 'N/A')
        eval_loss = entry.get('eval_loss', 'N/A')
        print(f"  Step {step}: train_loss={loss}, eval_loss={eval_loss}")

print("\n" + "="*60)
print("PROCHAINES ÉTAPES")
print("="*60)
print("""
1. Évaluer le modèle sur le test set (cassava_test.json)
2. Analyser la matrice de confusion par classe
3. Vérifier les performances sur CBB (classe rare)
4. Si résultats satisfaisants (>85% accuracy):
   - Passer au Stage 2: Conversations multi-tour
   - Créer notebook 02_train_stage2_cassava.ipynb
5. Si résultats insuffisants:
   - Analyser les erreurs
   - Ajuster hyperparamètres
   - Ré-entraîner avec plus d'époques

Fichiers générés:
  - {}
  - {}
  - {}
""".format(final_model_path, lora_adapters_path, CHECKPOINT_DIR / 'training_metrics.json'))

print("="*60)

## 11. Test Rapide (Optionnel)

Test rapide du modèle entraîné sur quelques images.

In [ ]:
# Code de test rapide (à compléter si nécessaire)
print("Test rapide non implémenté dans ce notebook.")
print("Utilisez le notebook d'évaluation pour des tests complets.")